In [12]:
import torch
import pandas as pd
from datasets import Dataset
from unsloth import FastModel 
from unsloth import is_bfloat16_supported
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments, EarlyStoppingCallback
from unsloth.chat_templates import standardize_data_formats, train_on_responses_only, get_chat_template
from sklearn.model_selection import train_test_split

lr = 5e-5
num_epochs = 3
batch_size = 32
weight_decay= 1e-3
num_beams   = 4
max_seq_length = 500 # Supports RoPE Scaling interally, so choose any!
seed=14
checkpoint   = "unsloth/gemma-3-27b-it-bnb-4bit"



#https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Gemma3_(4B).ipynb

df = pd.concat([pd.read_csv("./data/ustance/final_cino.csv", sep = "\t"),
                pd.read_csv("./data/ustance/final_cl.csv", sep = "\t"),
                pd.read_csv("./data/ustance/final_igreja.csv", sep = "\t")])

train, test = train_test_split(df, test_size=0.2, random_state=seed)
train, valid = train_test_split(train, test_size=0.1, random_state=seed)

#Text: é o texto original do autor
#generated_Text: é o texto generico (t)
dataset_train = Dataset.from_dict({"alvo": train["Text"].tolist(), "origem": train["generated_Text"].tolist()})
dataset_test = Dataset.from_dict({"alvo": test["Text"].tolist(), "origem": test["generated_Text"].tolist()})
dataset_valid = Dataset.from_dict({"alvo": valid["Text"].tolist(), "origem": valid["generated_Text"].tolist()})


In [11]:


model, tokenizer = FastModel.from_pretrained(
    model_name = checkpoint,
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # SHould leave on always!

    r = 16,           # Larger = higher accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = seed,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)

==((====))==  Unsloth 2025.3.19: Fast Gemma3 patching. Transformers: 4.50.3.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 22.488 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 4/4 [00:08<00:00,  2.11s/it]
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Unsloth: Making `model.base_model.model.language_model.model` require gradients


In [13]:

instrucao = """Reescreva o texto fornecido adotando o estilo de escrita do autor-alvo, mantendo inalterados o significado, a estrutura sintática e as informações do texto original. A tarefa deve preservar o conteúdo semântico, mas aplicar as características estilísticas do autor-alvo — como vocabulário, tom, ritmo, construção de frases e preferências linguísticas. Tanto o texto original quanto o texto reescrito estão em português. Forneça apenas uma versão reescrita, sem comentários adicionais."""

prompt_humano = """### Tarefa:
Transferência de Estilo Autoral.
### Instrução:
 {instructions}
### Texto original:
 {input_text}
"""
resposta_gpt = """### Texto reescrito no estilo do autor-alvo:
 {rewritten_text}
"""

def formatting_prompts_func(examples):
    instructions = instrucao
    inputs       = examples["origem"]
    outputs      = examples["alvo"]
    texts = []
    for input, output in zip(inputs, outputs):
        
        message = [
        {
        "from": "human",
        "value": prompt_humano.format(instructions=instructions, input_text=input)
        },
        {
            "from" : "gpt",
            "value" : resposta_gpt.format(rewritten_text=output),
        }]
        texts.append(message)
        

    return {"conversations": texts}

def apply_chat_template(example):
    texts = tokenizer.apply_chat_template(example["conversations"])
    
    return { "text" : texts }

In [14]:

print(dataset_train[100])
dataset_train = dataset_train.map(formatting_prompts_func, batched = True)
dataset_train = standardize_data_formats(dataset_train)
dataset_train = dataset_train.map(apply_chat_template, batched = True)

dataset_test = dataset_test.map(formatting_prompts_func, batched = True)
dataset_test = standardize_data_formats(dataset_test)
dataset_test = dataset_test.map(apply_chat_template, batched = True)

dataset_valid = dataset_valid.map(formatting_prompts_func, batched = True)
dataset_valid = standardize_data_formats(dataset_valid)
dataset_valid = dataset_valid.map(apply_chat_template, batched = True)




{'alvo': 'Coronavac n ta sendo usado nem na china kkkkkk , foi aprovado da sinopharm', 'origem': 'A Coronavac não está sendo utilizada nem na China, hahahaha, foi a vacina da Sinopharm que foi aprovada.'}


Map: 100%|██████████| 623/623 [00:00<00:00, 25859.76 examples/s]


In [15]:
print(dataset_train[100])

{'alvo': 'Coronavac n ta sendo usado nem na china kkkkkk , foi aprovado da sinopharm', 'origem': 'A Coronavac não está sendo utilizada nem na China, hahahaha, foi a vacina da Sinopharm que foi aprovada.', 'conversations': [{'content': '### Tarefa:\nTransferência de Estilo Autoral.\n### Instrução:\n Reescreva o texto fornecido adotando o estilo de escrita do autor-alvo, mantendo inalterados o significado, a estrutura sintática e as informações do texto original. A tarefa deve preservar o conteúdo semântico, mas aplicar as características estilísticas do autor-alvo — como vocabulário, tom, ritmo, construção de frases e preferências linguísticas. Tanto o texto original quanto o texto reescrito estão em português. Forneça apenas uma versão reescrita, sem comentários adicionais.\n### Texto original:\n A Coronavac não está sendo utilizada nem na China, hahahaha, foi a vacina da Sinopharm que foi aprovada.\n', 'role': 'user'}, {'content': '### Texto reescrito no estilo do autor-alvo:\n Corona

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset_train,
    eval_dataset = dataset_valid, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 5,
        num_train_epochs = num_epochs, # Set this for 1 full training run.
        learning_rate = 2e-5, # Reduce to 2e-5 for long training runs
        logging_steps = 0.1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = seed,
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=24): 100%|██████████| 7783/7783 [00:15<00:00, 504.34 examples/s]


In [9]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

Map (num_proc=24): 100%|██████████| 7783/7783 [00:00<00:00, 9343.29 examples/s]


In [10]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

'<bos><bos><start_of_turn>user\n### Tarefa:\nTransferência de Estilo Autoral.\n### Instrução:\n Reescreva o texto fornecido adotando o estilo de escrita do autor-alvo, mantendo inalterados o significado, a estrutura sintática e as informações do texto original. A tarefa deve preservar o conteúdo semântico, mas aplicar as características estilísticas do autor-alvo — como vocabulário, tom, ritmo, construção de frases e preferências linguísticas. Tanto o texto original quanto o texto reescrito estão em português. Forneça apenas uma versão reescrita, sem comentários adicionais.\n### Texto original:\n Um amigo meu que era ateu se tornou criminoso e agora acredita em Deus hahahaha, o crime está convertendo mais pessoas do que a igreja.<end_of_turn>\n<start_of_turn>model\n### Texto reescrito no estilo do autor-alvo:\n meu amigo era ateu, virou bandido agora acredita em Deus kkkkkkkk, a vida do crime convertendo mais que a igreja<end_of_turn>\n'

In [11]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

'                                                                                                                                                                             ### Texto reescrito no estilo do autor-alvo:\n meu amigo era ateu, virou bandido agora acredita em Deus kkkkkkkk, a vida do crime convertendo mais que a igreja<end_of_turn>\n'

In [12]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,783 | Num Epochs = 3 | Total steps = 2,919
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 113,516,544/27,000,000,000 (0.42% trained)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,4.768500
2,3.811600
3,4.284800
4,4.694500
5,3.747100
6,4.177200
7,3.553500
8,3.768100
9,3.537900
10,3.226600


KeyboardInterrupt: 

In [ ]:

messages = [{
    "role": "user",
    "content": [{
        "type" : "text",
        "text" : prompt_humano.format(instructions=instrucao, input_text="Caro amigo, a mídia é bastante entusiasta da vacina Coronavac. Se dependesse dela, João Dória seria o Presidente e o coronavírus teria entrado no Brasil sem necessidade de aprovação da Anvisa. Algo semelhante ao que ocorre em São Paulo."),
    }]
}]
text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
)
outputs = model.generate(
    **tokenizer([text], return_tensors = "pt").to("cuda"),
    max_new_tokens = 200, # Increase for longer outputs!
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
)


['<bos><start_of_turn>user\n### Tarefa:\nTransferência de Estilo Autoral.\n### Instrução:\n Reescreva o texto fornecido adotando o estilo de escrita do autor-alvo, mantendo inalterados o significado, a estrutura sintática e as informações do texto original. A tarefa deve preservar o conteúdo semântico, mas aplicar as características estilísticas do autor-alvo — como vocabulário, tom, ritmo, construção de frases e preferências linguísticas. Tanto o texto original quanto o texto reescrito estão em português. Forneça apenas uma versão reescrita, sem comentários adicionais.\n### Texto original:\n Caro amigo, a mídia é bastante entusiasta da vacina Coronavac. Se dependesse dela, João Dória seria o Presidente e o coronavírus teria entrado no Brasil sem necessidade de aprovação da Anvisa. Algo semelhante ao que ocorre em São Paulo.<end_of_turn>\n<start_of_turn>model\n Meu caro, a imprensa tem demonstrado um certo enlevo pela Coronavac. Se fosse por ela, João Dória já estaria na Presidência da

In [14]:
tokenizer.batch_decode(outputs)

['<bos><start_of_turn>user\n### Tarefa:\nTransferência de Estilo Autoral.\n### Instrução:\n Reescreva o texto fornecido adotando o estilo de escrita do autor-alvo, mantendo inalterados o significado, a estrutura sintática e as informações do texto original. A tarefa deve preservar o conteúdo semântico, mas aplicar as características estilísticas do autor-alvo — como vocabulário, tom, ritmo, construção de frases e preferências linguísticas. Tanto o texto original quanto o texto reescrito estão em português. Forneça apenas uma versão reescrita, sem comentários adicionais.\n### Texto original:\n Caro amigo, a mídia é bastante entusiasta da vacina Coronavac. Se dependesse dela, João Dória seria o Presidente e o coronavírus teria entrado no Brasil sem necessidade de aprovação da Anvisa. Algo semelhante ao que ocorre em São Paulo.<end_of_turn>\n<start_of_turn>model\n Meu caro, a imprensa tem demonstrado um certo enlevo pela Coronavac. Se fosse por ela, João Dória já estaria na Presidência da